# Notebook 1: 데이터 준비

## 목표
1. Zenodo10K에서 PPTX 5,000개 다운로드
2. 슬라이드별 썸네일 이미지 생성
3. 슬라이드 역할 약한 라벨(weak label) 자동 생성
4. Google Drive에 저장

## 예상 소요 시간
- 다운로드: 30~60분
- 썸네일 생성: 60~90분
- 라벨 생성: 10분

## 0. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = '/content/drive/MyDrive/dadeum_ml'
PPTX_DIR = f'{BASE_DIR}/pptx'
SLIDES_DIR = f'{BASE_DIR}/slides'
LABELS_DIR = f'{BASE_DIR}/labels'
MODELS_DIR = f'{BASE_DIR}/models'

for d in [PPTX_DIR, SLIDES_DIR, LABELS_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

print('디렉토리 구조 생성 완료')
print(f'BASE_DIR: {BASE_DIR}')

## 1. 패키지 설치

In [ ]:
!pip install -q datasets python-pptx pillow tqdm pandas huggingface_hub
!apt-get install -q libreoffice
print('설치 완료')

## 2. Zenodo10K PPTX 다운로드

Hugging Face의 `Forceless/Zenodo10K` 데이터셋에서 PPTX 파일을 다운로드한다.  
전체 10,448개 중 5,000개만 샘플링.

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download
from datasets import load_dataset
import pandas as pd
from tqdm import tqdm
import requests
import json

# 데이터셋 메타데이터 로드 (파일 URL 목록)
print('Zenodo10K 메타데이터 로딩 중...')
ds = load_dataset('Forceless/Zenodo10K', split='train', streaming=True)

# 5000개 샘플링
TARGET = 5000
samples = []
for item in tqdm(ds, total=TARGET, desc='메타데이터 수집'):
    samples.append(item)
    if len(samples) >= TARGET:
        break

print(f'수집된 샘플 수: {len(samples)}')
print(f'첫 번째 샘플 키: {list(samples[0].keys())}')

In [ ]:
# 첫 번째 샘플 구조 확인
import pprint
pprint.pprint(samples[0])

In [ ]:
# PPTX 파일 다운로드
# 데이터셋 구조에 따라 URL 키 이름이 다를 수 있음 — 위 출력 확인 후 조정
import urllib.request
from pathlib import Path

downloaded = []
failed = []

for i, sample in enumerate(tqdm(samples, desc='PPTX 다운로드')):
    # 키 이름 확인 후 수정 (url, file_url, download_url 등)
    url = sample.get('url') or sample.get('file_url') or sample.get('download_url')
    if url is None:
        # 바이트 데이터로 직접 제공되는 경우
        content = sample.get('content') or sample.get('file') or sample.get('bytes')
        if content:
            fname = f'{PPTX_DIR}/deck_{i:05d}.pptx'
            with open(fname, 'wb') as f:
                f.write(content if isinstance(content, bytes) else content.encode())
            downloaded.append(fname)
        continue

    fname = f'{PPTX_DIR}/deck_{i:05d}.pptx'
    if Path(fname).exists():
        downloaded.append(fname)
        continue

    try:
        urllib.request.urlretrieve(url, fname)
        downloaded.append(fname)
    except Exception as e:
        failed.append((i, str(e)))

print(f'다운로드 성공: {len(downloaded)}개')
print(f'다운로드 실패: {len(failed)}개')

# 진행 상황 저장
with open(f'{LABELS_DIR}/downloaded_files.json', 'w') as f:
    json.dump(downloaded, f)

## 3. 슬라이드 썸네일 생성

LibreOffice headless로 각 PPTX를 PNG로 변환한다.  
슬라이드별로 `slides/{deck_id}/slide_{N:03d}.png` 형태로 저장.

In [ ]:
import subprocess
import tempfile
import shutil
from pathlib import Path
from PIL import Image
import io

def pptx_to_thumbnails(pptx_path: str, output_dir: str, size: int = 224) -> list[str]:
    """
    PPTX → 슬라이드별 PNG (224x224 리사이즈)
    EfficientNet 입력 크기에 맞춤
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    with tempfile.TemporaryDirectory() as tmpdir:
        result = subprocess.run(
            ['libreoffice', '--headless', '--norestore',
             '--convert-to', 'png', '--outdir', tmpdir, pptx_path],
            capture_output=True, text=True, timeout=120
        )

        png_files = sorted(Path(tmpdir).glob('*.png'))
        if not png_files:
            return []

        saved = []
        for i, png_path in enumerate(png_files):
            img = Image.open(png_path).convert('RGB')
            img = img.resize((size, size), Image.LANCZOS)
            out_path = f'{output_dir}/slide_{i:03d}.png'
            img.save(out_path, 'PNG', optimize=True)
            saved.append(out_path)

        return saved


# 전체 변환 실행
all_slide_paths = {}  # deck_id → [slide_path, ...]
convert_errors = []

for pptx_path in tqdm(downloaded, desc='썸네일 생성'):
    deck_id = Path(pptx_path).stem  # deck_00001
    out_dir = f'{SLIDES_DIR}/{deck_id}'

    # 이미 변환된 경우 스킵
    if Path(out_dir).exists() and len(list(Path(out_dir).glob('*.png'))) > 0:
        slides = sorted([str(p) for p in Path(out_dir).glob('*.png')])
        all_slide_paths[deck_id] = slides
        continue

    try:
        slides = pptx_to_thumbnails(pptx_path, out_dir)
        if slides:
            all_slide_paths[deck_id] = slides
        else:
            convert_errors.append(deck_id)
    except Exception as e:
        convert_errors.append(deck_id)

total_slides = sum(len(v) for v in all_slide_paths.values())
print(f'변환 성공: {len(all_slide_paths)}개 덱, 총 {total_slides}장 슬라이드')
print(f'변환 실패: {len(convert_errors)}개')

## 4. 약한 라벨(Weak Label) 자동 생성

슬라이드 위치와 python-pptx 특징으로 역할을 자동 분류.

| 역할 | 규칙 |
|---|---|
| 0: 표지 | 첫 번째 슬라이드 |
| 1: 섹션헤더 | 텍스트 짧음(< 15단어) + 전체의 10~40% 위치 |
| 2: 본문 | 텍스트 많음(≥ 15단어) |
| 3: 도표 | 이미지 비율 높음(> 40%) |
| 4: 마무리 | 마지막 슬라이드 |


In [ ]:
from pptx import Presentation
from pptx.util import Emu
import numpy as np

ROLE_COVER = 0
ROLE_SECTION = 1
ROLE_BODY = 2
ROLE_VISUAL = 3
ROLE_CLOSING = 4
ROLE_NAMES = ['표지', '섹션헤더', '본문', '도표/시각자료', '마무리']


def get_slide_features(slide, slide_width, slide_height):
    """슬라이드에서 라벨링에 필요한 특징 추출"""
    total_area = slide_width * slide_height
    word_count = 0
    image_area = 0

    for shape in slide.shapes:
        if shape.has_text_frame:
            for para in shape.text_frame.paragraphs:
                for run in para.runs:
                    word_count += len(run.text.split())
        if shape.shape_type == 13:  # MSO_SHAPE_TYPE.PICTURE
            image_area += shape.width * shape.height

    image_ratio = image_area / total_area if total_area > 0 else 0
    return word_count, image_ratio


def assign_weak_label(slide_idx: int, total_slides: int,
                       word_count: int, image_ratio: float) -> int:
    """위치 + 특징 기반 약한 라벨 할당"""
    position_ratio = slide_idx / max(total_slides - 1, 1)

    if slide_idx == 0:
        return ROLE_COVER
    if slide_idx == total_slides - 1:
        return ROLE_CLOSING
    if image_ratio > 0.40:
        return ROLE_VISUAL
    if word_count < 15 and 0.05 < position_ratio < 0.85:
        return ROLE_SECTION
    return ROLE_BODY


# 전체 덱에 대해 라벨 생성
records = []

for pptx_path in tqdm(downloaded, desc='약한 라벨 생성'):
    deck_id = Path(pptx_path).stem
    if deck_id not in all_slide_paths:
        continue

    try:
        prs = Presentation(pptx_path)
        n = len(prs.slides)
        W = prs.slide_width
        H = prs.slide_height

        for i, slide in enumerate(prs.slides):
            slide_png = f'{SLIDES_DIR}/{deck_id}/slide_{i:03d}.png'
            if not Path(slide_png).exists():
                continue

            word_count, image_ratio = get_slide_features(slide, W, H)
            label = assign_weak_label(i, n, word_count, image_ratio)

            records.append({
                'deck_id': deck_id,
                'slide_idx': i,
                'total_slides': n,
                'position_ratio': i / max(n - 1, 1),
                'word_count': word_count,
                'image_ratio': round(image_ratio, 4),
                'weak_label': label,
                'role_name': ROLE_NAMES[label],
                'image_path': slide_png,
            })
    except Exception:
        continue

df = pd.DataFrame(records)
print(f'총 슬라이드 수: {len(df)}')
print('\n역할별 분포:')
print(df['role_name'].value_counts())

In [ ]:
# 라벨 저장
label_path = f'{LABELS_DIR}/weak_labels.csv'
df.to_csv(label_path, index=False)
print(f'저장 완료: {label_path}')

# 샘플 시각화
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for role_id, ax in enumerate(axes):
    subset = df[df['weak_label'] == role_id]
    if len(subset) == 0:
        ax.set_title(f'{ROLE_NAMES[role_id]}\n(샘플 없음)')
        ax.axis('off')
        continue
    sample = subset.sample(1).iloc[0]
    img = mpimg.imread(sample['image_path'])
    ax.imshow(img)
    ax.set_title(f'{ROLE_NAMES[role_id]}\n({len(subset)}장)', fontsize=10)
    ax.axis('off')

plt.suptitle('역할별 슬라이드 샘플', fontsize=14)
plt.tight_layout()
plt.savefig(f'{LABELS_DIR}/role_samples.png', dpi=100)
plt.show()
print('시각화 저장 완료')

## 5. HMM 학습용 시퀀스 데이터 생성

덱별 역할 시퀀스를 추출해서 Notebook 3 HMM 학습에 사용할 데이터 저장.

In [ ]:
# 덱별 역할 시퀀스 추출
sequences = []

for deck_id, group in df.groupby('deck_id'):
    group = group.sort_values('slide_idx')
    seq = group['weak_label'].tolist()
    if len(seq) >= 3:  # 최소 3장 이상인 덱만
        sequences.append({'deck_id': deck_id, 'sequence': seq, 'length': len(seq)})

seq_df = pd.DataFrame(sequences)
seq_df.to_csv(f'{LABELS_DIR}/sequences.csv', index=False)

print(f'시퀀스 수: {len(seq_df)}')
print(f'평균 덱 길이: {seq_df["length"].mean():.1f}장')
print(f'최대 덱 길이: {seq_df["length"].max()}장')

# 가장 흔한 시퀀스 패턴 확인
from collections import Counter
short_seqs = seq_df[seq_df['length'] <= 10]['sequence'].tolist()
pattern_counts = Counter([tuple(s) for s in short_seqs])
print('\n가장 흔한 시퀀스 패턴 (10장 이하):')
for pattern, count in pattern_counts.most_common(5):
    readable = ' → '.join([ROLE_NAMES[r] for r in pattern])
    print(f'  {readable}  ({count}개)')

In [ ]:
print('=== Notebook 1 완료 ===')
print(f'PPTX 다운로드: {len(downloaded)}개')
print(f'슬라이드 이미지: {len(df)}장')
print(f'약한 라벨 CSV: {label_path}')
print(f'시퀀스 CSV: {LABELS_DIR}/sequences.csv')
print('\nNotebook 2 (CNN 역할 분류기)로 이동하세요.')